# 34. The Hazard & IMDG Segregation Problem

## Tier 1: The Pen & Paper Method (Mixed-Integer Programming Formulation)

### Goal
Formulate and solve the IMDG segregation problem using mixed-integer programming to ensure optimal container placement while satisfying all dangerous goods segregation requirements.

### Key assumptions
- Each container must be assigned to exactly one position
- Each position can hold at most one container
- Segregation distance requirements must be satisfied between incompatible containers
- Same-hold restrictions apply for certain incompatible material combinations

### Approach (step-by-step)
1. Define the mathematical model with sets, parameters, and decision variables
2. Implement the mixed-integer programming formulation using pulp
3. Create a concrete example with 4 containers and 6 positions
4. Solve the optimization problem and analyze results
5. Visualize the container placement solution

### What to look for in the results
- Optimal container placement with zero segregation violations
- Proper assignment of containers to positions respecting all constraints
- Total cost minimization considering violation penalties and position preferences

### Concrete example (from the source)
4 containers with IMDG classes:
- Container A (Class 1.4S explosives)
- Container B (Class 3 flammable liquid)
- Container C (Class 8.1 corrosive acid)
- Container D (Class 2.1 flammable gas)

Segregation requirements:
- A-B: "Away From" (3m)
- A-C: "Separated From" (different holds)
- A-D: "Away From" (3m)
- B-C: "Away From" (3m)
- B-D: No restriction
- C-D: "Away From" (3m)

6 available positions across 2 holds.

In [1]:
# Import required packages for mixed-integer programming
import pulp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple
import warnings
warnings.filterwarnings('ignore')

# Set up visualization style
plt.style.use('default')
sns.set_palette("husl")

In [2]:
# Define data structures for IMDG segregation problem
from dataclasses import dataclass
from typing import List, Dict, Tuple

@dataclass
class Container:
    """Represents a container with dangerous goods"""
    id: str
    imdg_class: str
    weight: float = 20.0  # TEU weight

@dataclass
class Position:
    """Represents a stowage position on the vessel"""
    id: str
    hold: int
    x: float  # horizontal position
    y: float  # vertical position
    z: float  # depth position

@dataclass
class SegregationRequirement:
    """Represents segregation requirements between container classes"""
    class1: str
    class2: str
    requirement: str  # "Away From", "Separated From", "No restriction"
    min_distance: float  # minimum distance in meters
    different_hold: bool = False  # requires different holds

In [ ]:
# Create the concrete example from the source
def create_concrete_example():
    """Create the concrete example with 4 containers and 6 positions"""
    
    # Define containers
    containers = [
        Container("A", "1.4S"),  # explosives
        Container("B", "3"),     # flammable liquid
        Container("C", "8.1"),   # corrosive acid
        Container("D", "2.1")    # flammable gas
    ]
    
    # Define positions (3 positions per hold, 2 holds)
    positions = [
        # Hold 1 positions
        Position("H1_P1", 1, 0, 0, 0),   # Position 1 in Hold 1
        Position("H1_P2", 1, 3, 0, 0),   # Position 2 in Hold 1 (3m from P1)
        Position("H1_P3", 1, 6, 0, 0),   # Position 3 in Hold 1 (6m from P1)
        # Hold 2 positions
        Position("H2_P1", 2, 0, 0, 0),   # Position 1 in Hold 2
        Position("H2_P2", 2, 3, 0, 0),   # Position 2 in Hold 2
        Position("H2_P3", 2, 6, 0, 0),   # Position 3 in Hold 2
    ]
    
    # Define segregation requirements
    segregation_requirements = [
        SegregationRequirement("1.4S", "3", "Away From", 3.0),
        SegregationRequirement("1.4S", "8.1", "Separated From", 0.0, True),
        SegregationRequirement("1.4S", "2.1", "Away From", 3.0),
        SegregationRequirement("3", "8.1", "Away From", 3.0),
        SegregationRequirement("3", "2.1", "No restriction", 0.0),
        SegregationRequirement("8.1", "2.1", "Away From", 3.0),
    ]
    
    return containers, positions, segregation_requirements

# Create the example
containers, positions, seg_reqs = create_concrete_example()

print("Containers:")
for c in containers:
    print(f"  {c.id}: Class {c.imdg_class}")

print("\nPositions:")
for p in positions:
    print(f"  {p.id}: Hold {p.hold}, Position ({p.x}, {p.y}, {p.z})")

print("\nSegregation Requirements:")
for req in seg_reqs:
    print(f"  {req.class1} - {req.class2}: {req.requirement} (min: {req.min_distance}m, different_hold: {req.different_hold})")

In [ ]:
# Calculate distance matrix between positions
def calculate_distance_matrix(positions: List[Position]) -> Dict[Tuple[str, str], float]:
    """Calculate Euclidean distances between all position pairs"""
    distances = {}
    for i, pos1 in enumerate(positions):
        for j, pos2 in enumerate(positions):
            if i != j:
                # Calculate 3D Euclidean distance
                dist = np.sqrt((pos1.x - pos2.x)**2 + (pos1.y - pos2.y)**2 + (pos1.z - pos2.z)**2)
                distances[(pos1.id, pos2.id)] = dist
    return distances

# Check segregation compliance between two containers
def check_segregation(container1: Container, container2: Container, 
                     pos1: Position, pos2: Position, 
                     requirements: List[SegregationRequirement],
                     distances: Dict[Tuple[str, str], float]) -> bool:
    """Check if two containers in given positions satisfy segregation requirements"""
    
    # Find applicable requirement
    applicable_req = None
    for req in requirements:
        if ((req.class1 == container1.imdg_class and req.class2 == container2.imdg_class) or
            (req.class1 == container2.imdg_class and req.class2 == container1.imdg_class)):
            applicable_req = req
            break
    
    if applicable_req is None or applicable_req.requirement == "No restriction":
        return True
    
    # Check distance requirement
    distance = distances.get((pos1.id, pos2.id), 0)
    if distance < applicable_req.min_distance:
        return False
    
    # Check different hold requirement
    if applicable_req.different_hold and pos1.hold == pos2.hold:
        return False
    
    return True

# Calculate distance matrix
distance_matrix = calculate_distance_matrix(positions)

print("Distance matrix (meters):")
for (pos1_id, pos2_id), dist in distance_matrix.items():
    if dist <= 6.0:  # Show only reasonable distances
        print(f"  {pos1_id} - {pos2_id}: {dist:.1f}m")

In [ ]:
# Implement the mixed-integer programming model
def solve_imdg_segregation_mip(containers: List[Container], 
                              positions: List[Position],
                              requirements: List[SegregationRequirement],
                              distances: Dict[Tuple[str, str], float]) -> Dict:
    """Solve IMDG segregation problem using mixed-integer programming"""
    
    # Create the optimization problem
    prob = pulp.LpProblem("IMDG_Segregation", pulp.LpMinimize)
    
    # Decision variables
    # x[i][p] = 1 if container i is assigned to position p
    x = {}
    for i, container in enumerate(containers):
        for p, position in enumerate(positions):
            x[(i, p)] = pulp.LpVariable(f"x_{i}_{p}", cat='Binary')
    
    # y[i][j] = 1 if containers i and j violate segregation requirements
    y = {}
    for i in range(len(containers)):
        for j in range(len(containers)):
            if i < j:  # Only create for i < j to avoid duplicates
                y[(i, j)] = pulp.LpVariable(f"y_{i}_{j}", cat='Binary')
    
    # Big M penalty coefficient for violations
    M = 10000
    
    # Position preference weights (prefer positions in lower holds for stability)
    position_weights = {p.id: p.hold for p in positions}
    
    # Objective function: minimize violations + position preferences
    prob += (
        pulp.lpSum(M * y[(i, j)] for i in range(len(containers)) for j in range(len(containers)) if i < j) +
        pulp.lpSum(position_weights[p.id] * x[(i, p)] for i in range(len(containers)) for p, position in enumerate(positions))
    ), "Total_Cost"
    
    # Constraint 1: Each container must be assigned to exactly one position
    for i in range(len(containers)):
        prob += pulp.lpSum(x[(i, p)] for p in range(len(positions))) == 1, f"Container_{i}_Assignment"
    
    # Constraint 2: Each position can hold at most one container
    for p in range(len(positions)):
        prob += pulp.lpSum(x[(i, p)] for i in range(len(containers))) <= 1, f"Position_{p}_Capacity"
    
    # Constraint 3: Segregation distance constraints
    for i in range(len(containers)):
        for j in range(len(containers)):
            if i < j:
                container1, container2 = containers[i], containers[j]
                
                # Find applicable requirement
                applicable_req = None
                for req in requirements:
                    if ((req.class1 == container1.imdg_class and req.class2 == container2.imdg_class) or
                        (req.class1 == container2.imdg_class and req.class2 == container1.imdg_class)):
                        applicable_req = req
                        break
                
                if applicable_req and applicable_req.requirement != "No restriction":
                    # Distance constraint
                    if applicable_req.min_distance > 0:
                        for p1 in range(len(positions)):
                            for p2 in range(len(positions)):
                                if p1 != p2:
                                    pos1, pos2 = positions[p1], positions[p2]
                                    dist = distances.get((pos1.id, pos2.id), 0)
                                    prob += (
                                        dist * x[(i, p1)] * x[(j, p2)] >= applicable_req.min_distance - M * y[(i, j)]
                                    ), f"Distance_{i}_{j}_{p1}_{p2}"
                    
                    # Different hold constraint
                    if applicable_req.different_hold:
                        for h in range(1, 3):  # Holds 1 and 2
                            hold_positions = [p for p, pos in enumerate(positions) if pos.hold == h]
                            prob += (
                                pulp.lpSum(x[(i, p)] for p in hold_positions) +
                                pulp.lpSum(x[(j, p)] for p in hold_positions) <= 1 + M * y[(i, j)]
                            ), f"Different_Hold_{i}_{j}_{h}"
    
    # Solve the problem
    print("Solving MIP optimization problem...")
    prob.solve(pulp.PULP_CBC_CMD(msg=False))
    
    # Extract results
    status = pulp.LpStatus[prob.status]
    objective_value = pulp.value(prob.objective)
    
    # Extract container assignments
    assignments = {}
    for i in range(len(containers)):
        for p in range(len(positions)):
            if pulp.value(x[(i, p)]) > 0.5:
                assignments[containers[i].id] = positions[p]
                break
    
    # Extract violations
    violations = []
    for i in range(len(containers)):
        for j in range(len(containers)):
            if i < j and pulp.value(y[(i, j)]) > 0.5:
                violations.append((containers[i].id, containers[j].id))
    
    return {
        'status': status,
        'objective_value': objective_value,
        'assignments': assignments,
        'violations': violations,
        'problem': prob
    }

In [ ]:
# Solve the concrete example
result = solve_imdg_segregation_mip(containers, positions, seg_reqs, distance_matrix)

print(f"Optimization Status: {result['status']}")
print(f"Objective Value: {result['objective_value']:.2f}")
print(f"Number of Violations: {len(result['violations'])}")

if result['violations']:
    print("Segregation Violations:")
    for v in result['violations']:
        print(f"  {v[0]} - {v[1]}")
else:
    print("No segregation violations - all constraints satisfied!")

print("\nContainer Assignments:")
for container_id, position in result['assignments'].items():
    container = next(c for c in containers if c.id == container_id)
    print(f"  {container_id} (Class {container.imdg_class}) -> {position.id} (Hold {position.hold})")

In [ ]:
# Verify the solution manually
def verify_solution(assignments: Dict[str, Position], 
                   containers: List[Container],
                   requirements: List[SegregationRequirement],
                   distances: Dict[Tuple[str, str], float]) -> List[Dict]:
    """Verify that the solution satisfies all segregation requirements"""
    
    verification_results = []
    
    for i, container1 in enumerate(containers):
        for j, container2 in enumerate(containers):
            if i < j:
                pos1 = assignments[container1.id]
                pos2 = assignments[container2.id]
                
                # Find applicable requirement
                applicable_req = None
                for req in requirements:
                    if ((req.class1 == container1.imdg_class and req.class2 == container2.imdg_class) or
                        (req.class1 == container2.imdg_class and req.class2 == container1.imdg_class)):
                        applicable_req = req
                        break
                
                if applicable_req:
                    is_compliant = check_segregation(container1, container2, pos1, pos2, requirements, distances)
                    actual_distance = distances.get((pos1.id, pos2.id), 0)
                    
                    verification_results.append({
                        'container1': container1.id,
                        'class1': container1.imdg_class,
                        'container2': container2.id,
                        'class2': container2.imdg_class,
                        'requirement': applicable_req.requirement,
                        'min_distance': applicable_req.min_distance,
                        'different_hold': applicable_req.different_hold,
                        'pos1': pos1.id,
                        'pos2': pos2.id,
                        'actual_distance': actual_distance,
                        'pos1_hold': pos1.hold,
                        'pos2_hold': pos2.hold,
                        'compliant': is_compliant
                    })
    
    return verification_results

# Verify the solution
verification = verify_solution(result['assignments'], containers, seg_reqs, distance_matrix)

print("Solution Verification:")
print("=" * 80)
for v in verification:
    status = "✓" if v['compliant'] else "✗"
    print(f"{status} {v['container1']}({v['class1']}) - {v['container2']}({v['class2']}): {v['requirement']}")
    print(f"   Positions: {v['pos1']} (Hold {v['pos1_hold']}) - {v['pos2']} (Hold {v['pos2_hold']})")
    print(f"   Distance: {v['actual_distance']:.1f}m (min: {v['min_distance']}m)")
    if v['different_hold']:
        print(f"   Hold requirement: Different holds ({'✓' if v['pos1_hold'] != v['pos2_hold'] else '✗'})")
    print()

In [ ]:
# Visualize the container placement solution
def visualize_solution(assignments: Dict[str, Position], 
                      containers: List[Container],
                      positions: List[Position]):
    """Create a visualization of the container placement solution"""
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Plot 1: Hold layout
    hold1_positions = [p for p in positions if p.hold == 1]
    hold2_positions = [p for p in positions if p.hold == 2]
    
    # Hold 1 visualization
    ax1.set_title('Hold 1 Layout', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Horizontal Position (m)')
    ax1.set_ylabel('Vertical Position (m)')
    ax1.grid(True, alpha=0.3)
    
    for pos in hold1_positions:
        # Find container at this position
        container_id = None
        container_class = None
        for cid, assigned_pos in assignments.items():
            if assigned_pos.id == pos.id:
                container_id = cid
                container_class = next(c.imdg_class for c in containers if c.id == cid)
                break
        
        if container_id:
            ax1.scatter(pos.x, pos.y, s=500, c='red', marker='s', alpha=0.7)
            ax1.text(pos.x, pos.y, f'{container_id}\n{container_class}', 
                    ha='center', va='center', fontsize=10, fontweight='bold')
        else:
            ax1.scatter(pos.x, pos.y, s=500, c='lightgray', marker='s', alpha=0.5)
            ax1.text(pos.x, pos.y, 'Empty', ha='center', va='center', fontsize=9)
    
    # Hold 2 visualization
    ax2.set_title('Hold 2 Layout', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Horizontal Position (m)')
    ax2.set_ylabel('Vertical Position (m)')
    ax2.grid(True, alpha=0.3)
    
    for pos in hold2_positions:
        # Find container at this position
        container_id = None
        container_class = None
        for cid, assigned_pos in assignments.items():
            if assigned_pos.id == pos.id:
                container_id = cid
                container_class = next(c.imdg_class for c in containers if c.id == cid)
                break
        
        if container_id:
            ax2.scatter(pos.x, pos.y, s=500, c='red', marker='s', alpha=0.7)
            ax2.text(pos.x, pos.y, f'{container_id}\n{container_class}', 
                    ha='center', va='center', fontsize=10, fontweight='bold')
        else:
            ax2.scatter(pos.x, pos.y, s=500, c='lightgray', marker='s', alpha=0.5)
            ax2.text(pos.x, pos.y, 'Empty', ha='center', va='center', fontsize=9)
    
    plt.suptitle('IMDG Segregation - Optimal Container Placement', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Create segregation compliance table
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.axis('tight')
    ax.axis('off')
    
    # Prepare table data
    table_data = [['Container Pair', 'IMDG Classes', 'Requirement', 'Positions', 'Distance', 'Compliance']]
    colors = [['lightgray', 'lightgray', 'lightgray', 'lightgray', 'lightgray', 'lightgray']]
    
    for v in verification:
        compliance_symbol = "✓" if v['compliant'] else "✗"
        compliance_color = 'lightgreen' if v['compliant'] else 'lightcoral'
        
        table_data.append([
            f"{v['container1']} - {v['container2']}",
            f"{v['class1']} - {v['class2']}",
            v['requirement'],
            f"{v['pos1']} - {v['pos2']}",
            f"{v['actual_distance']:.1f}m",
            compliance_symbol
        ])
        colors.append(['white', 'white', 'white', 'white', 'white', compliance_color])
    
    table = ax.table(cellText=table_data, cellColours=colors, 
                    cellLoc='center', loc='center', colWidths=[0.15, 0.15, 0.2, 0.2, 0.1, 0.1])
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 2)
    
    # Style header row
    for i in range(6):
        table[(0, i)].set_facecolor('#4472C4')
        table[(0, i)].set_text_props(weight='bold', color='white')
    
    plt.title('IMDG Segregation Compliance Table', fontsize=14, fontweight='bold', pad=20)
    plt.show()

# Visualize the solution
visualize_solution(result['assignments'], containers, positions)

In [ ]:
# Sensitivity analysis - what if we change the minimum distance requirement?
def sensitivity_analysis():
    """Perform sensitivity analysis on minimum distance requirements"""
    
    # Test different minimum distance scenarios
    distance_multipliers = [0.5, 1.0, 1.5, 2.0]
    results = []
    
    for multiplier in distance_multipliers:
        # Modify segregation requirements
        modified_reqs = []
        for req in seg_reqs:
            modified_req = SegregationRequirement(
                req.class1, req.class2, req.requirement,
                req.min_distance * multiplier, req.different_hold
            )
            modified_reqs.append(modified_req)
        
        # Solve with modified requirements
        result_modified = solve_imdg_segregation_mip(containers, positions, modified_reqs, distance_matrix)
        
        results.append({
            'multiplier': multiplier,
            'objective': result_modified['objective_value'],
            'violations': len(result_modified['violations']),
            'status': result_modified['status']
        })
    
    # Create sensitivity analysis visualization
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Plot 1: Objective value vs distance multiplier
    multipliers = [r['multiplier'] for r in results]
    objectives = [r['objective'] for r in results]
    
    ax1.plot(multipliers, objectives, 'bo-', linewidth=2, markersize=8)
    ax1.set_xlabel('Distance Multiplier')
    ax1.set_ylabel('Objective Value')
    ax1.set_title('Impact of Distance Requirements on Solution Quality')
    ax1.grid(True, alpha=0.3)
    ax1.set_xticks(multipliers)
    
    # Plot 2: Violations vs distance multiplier
    violations = [r['violations'] for r in results]
    
    ax2.bar(range(len(distance_multipliers)), violations, color=['green' if v == 0 else 'red' for v in violations])
    ax2.set_xlabel('Distance Multiplier')
    ax2.set_ylabel('Number of Violations')
    ax2.set_title('Segregation Violations vs Distance Requirements')
    ax2.set_xticks(range(len(distance_multipliers)))
    ax2.set_xticklabels([f'{m}x' for m in distance_multipliers])
    ax2.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for i, v in enumerate(violations):
        ax2.text(i, v + 0.1, str(v), ha='center', va='bottom', fontweight='bold')
    
    plt.suptitle('Sensitivity Analysis: Impact of Distance Requirements', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Print detailed results
    print("Sensitivity Analysis Results:")
    print("=" * 60)
    for r in results:
        print(f"Distance Multiplier: {r['multiplier']}x")
        print(f"  Objective Value: {r['objective']:.2f}")
        print(f"  Violations: {r['violations']}")
        print(f"  Status: {r['status']}")
        print()

# Run sensitivity analysis
sensitivity_analysis()

### Why this Tier exists vs earlier Tiers
This is Tier 1, the foundational mathematical approach. It provides:
- **Optimal solutions** with provable optimality guarantees
- **Complete constraint satisfaction** through mathematical formulation
- **Baseline for comparison** with heuristic and metaheuristic methods
- **Understanding of problem structure** through mathematical modeling

### Pros / Cons vs this Tier
**Advantages:**
- Guarantees optimal solution
- Handles all constraints exactly
- Provides optimality gap information
- Reproducible and deterministic results

**Disadvantages:**
- Computationally expensive for large instances
- Requires specialized optimization software
- May not scale to real-world problem sizes
- Limited flexibility for dynamic changes

### When to use this Tier
- **Small to medium instances** where optimality is critical
- **Regulatory compliance** requiring provable constraint satisfaction
- **Benchmark development** for comparing other methods
- **Academic and research settings** where mathematical rigor is required
- **Safety-critical applications** where optimal solutions are mandatory